## Structured Output
Getting the structed output from the models, the strucvted are defined in differnet ways to get the output

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
BASE_URL = os.getenv('OPENAI_BASE_URL')

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-5.1",
    temperature=1.7,
    api_key=API_KEY,
    base_url=BASE_URL
)

llm.invoke('Hi')

AIMessage(content='Hi! How can I help you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 19, 'prompt_tokens': 7, 'total_tokens': 26, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 6, 'engine_ttft_ms': 62, 'engine_ttlt_ms': 178, 'pre_inference_ms': 99, 'service_tbt_ms': 6, 'service_ttft_ms': 350, 'service_ttlt_ms': 463, 'total_duration_ms': 371, 'user_visible_ttft_ms': 251}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-DdbduAU3JIiqCERlDqcW41r5zuxKd', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e0cd6-c123-78a3-b3d2-d110dbd188aa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 19, 'total_toke

In [33]:
from langchain_core.prompts import PromptTemplate

review_template = PromptTemplate(
    template = 'User Review: {review} What is the sentiment, rating (1-5), also what is the suggesestion by the user If there is no suggestion, then keep it empty?'
)

review_chain = review_template | model

### TypedDict

In [11]:
from typing import TypedDict, List, Literal, Optional

In [ ]:
class Person(TypedDict):
    name: str
    age: int
    city: str

model = llm.with_structured_output(Person)
response = model.invoke('What is your name, age, and city?')

print(response)
print(type(response), response.keys())

{'name': 'ChatGPT', 'age': 0, 'city': 'Internet'}
<class 'dict'> dict_keys(['name', 'age', 'city'])


In the response we are getting the Dictonary object.

In [32]:
class Review(TypedDict):
    Sentiment: Literal['Positive', 'Negative', 'Neutral']
    rating: int = 3
    suggestions: Optional[str]
    summary: str

prompt = review_template.invoke({'review':"The product is good"})

model = llm.with_structured_output(Review)
response = model.invoke(prompt)

print(response)
print(type(response), response.keys())

{'Sentiment': 'Positive', 'rating': 4, 'suggestions': '', 'summary': 'The user feels the product is good, indicating a positive experience without specific suggestions.'}
<class 'dict'> dict_keys(['Sentiment', 'rating', 'suggestions', 'summary'])


In [34]:
review_chain.invoke({'review':"The product is deascent they should change the packing of it"})

{'Sentiment': 'Neutral',
 'rating': 3,
 'suggestions': 'They should change/improve the product packaging.',
 'summary': 'The product is decent but the user wants the packaging to be changed or improved.'}

In [36]:
# response = model.invoke("""
# I absolutely love this phone! The camera quality is outstanding, and the battery life lasts all day
# even with heavy use. The user interface is sleek and intuitive, making it easy to navigate through apps and settings.
# Overall, this phone exceeds my expectations and I highly recommend it to anyone in the market for a new device.
# """)

response = review_chain.invoke("""
I absolutely love this phone! The camera quality is outstanding, and the battery life lasts all day
even with heavy use. The user interface is sleek and intuitive, making it easy to navigate through apps and settings.
Overall, this phone exceeds my expectations and I highly recommend it to anyone in the market for a new device.
""")

print(response)
print(type(response), response.keys())

{'Sentiment': 'Positive', 'rating': 5, 'suggestions': '', 'summary': 'The user is extremely satisfied with the phone, praising its outstanding camera quality, all-day battery life under heavy use, and sleek, intuitive user interface, and highly recommends it to others.'}
<class 'dict'> dict_keys(['Sentiment', 'rating', 'suggestions', 'summary'])


### DataClass

In [20]:
from dataclasses import dataclass

In [ ]:
@dataclass
class ReviewDataClass:
    Sentiment: Literal['Positive', 'Negative', 'Neutral']
    rating: int 
    suggestions: Optional[str]
    summary: str

model = llm.with_structured_output(ReviewDataClass)
# response = model.invoke(prompt)

response = review_chain.invoke({
    'review': "The product is deascent they should change the packing of it"
})

print(response)
print(type(response))
print(response.keys())

{'Sentiment': 'Neutral', 'rating': 3, 'suggestions': 'They should change/improve the product packaging.', 'summary': 'The product is decent but the user wants the packaging to be changed or improved.'}
<class 'dict'>
dict_keys(['Sentiment', 'rating', 'suggestions', 'summary'])


### Type Annotation

In [38]:
from typing import Dict, List, Literal, Optional

In [57]:
class DataClassReview:
    summary: str
    sentiment: Literal['positive', 'negative', 'neutral']

In [60]:
model = llm.with_structured_output(DataClassReview)
review_chain = review_template | model
response = review_chain.invoke({'review':"The product is deascent they should change the packing of it"})

print(response)
print(type(response))
print(response.keys())

{}
<class 'dict'>
dict_keys([])


### Pydantic

In [80]:
from pydantic import BaseModel, Field, EmailStr
from typing import Optional, List, Literal

In [65]:
class Studnet(BaseModel):
    name: str
    age: int
    city: str

In [66]:
person_template = PromptTemplate(
    template = 'Give me name, age, and city of the person form {country}?'
)

model = llm.with_structured_output(Studnet)

In [67]:
person_chain = person_template | model
response = person_chain.invoke({'country': 'India'})

print(response)
print(type(response))
# print(response.keys())

name='Raj Kumar' age=28 city='Mumbai'
<class '__main__.Studnet'>


In [84]:
class Address(BaseModel):
    street: str
    city: str
    state: str
    zip_code: str

class Person(BaseModel):
    name: str
    age: int
    address: Address
    salary: Optional[float] = Field(default=None, description="The salary of the person")
    email: EmailStr = Field(..., description="The email address of the person")

In [85]:
person_template = PromptTemplate(
    template = 'Give me details of a random hypothetical person from {country}? And give the salary on random basis in local currency'
)

model = llm.with_structured_output(Person)

In [86]:
person_chain = person_template | model
response = person_chain.invoke({'country': 'India'})

print(response)
print(type(response))
print(response.email)

name='Amit Deshpande' age=34 address=Address(street='402, Shanti Enclave Apartments, MG Road', city='Pune', state='Maharashtra', zip_code='411001') salary=93250.0 email='amit.deshpande34@example.in'
<class '__main__.Person'>
amit.deshpande34@example.in


In [87]:
person_chain = person_template | model
response = person_chain.invoke({'country': 'Sri Lanka'})

print(response)
print(type(response))
print(response.address)


name='Nimal Perera' age=34 address=Address(street='42 Galle Road', city='Colombo', state='Western Province', zip_code='00300') salary=185000.0 email='nimal.perera@example.lk'
<class '__main__.Person'>
street='42 Galle Road' city='Colombo' state='Western Province' zip_code='00300'


In [88]:
person_chain = person_template | model
response = person_chain.invoke({'country': 'USA'})

print(response)
print(type(response))
print(response.address)


name='Michael Anderson' age=34 address=Address(street='4829 Willow Creek Drive', city='Aurora', state='CO', zip_code='80013') salary=87250.75 email='michael.anderson34@example.com'
<class '__main__.Person'>
street='4829 Willow Creek Drive' city='Aurora' state='CO' zip_code='80013'


### JSON

In [89]:
json_schema = {
  "title": "Review",
  "type": "object",
  "properties": {
    "key_themes": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "Write down all the key themes discussed in the review in a list"
    },
    "summary": {
      "type": "string",
      "description": "A brief summary of the review"
    },
    "sentiment": {
      "type": "string",
      "enum": ["positive", "negative", "neutral"],
      "description": "Return sentiment of the review either negative, positive or neutral"
    },
    "pros": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the pros inside a list"
    },
    "cons": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the cons inside a list"
    },
    "name": {
      "type": ["string", "null"],
      "description": "Write the name of the reviewer"
    }
  },
  "required": ["key_themes", "summary", "sentiment"]
}


In [92]:
model = llm.with_structured_output(json_schema)
review_chain = review_template | model

response = review_chain.invoke({
    'review': """
I recently upgraded to the Samsung Galaxy S25 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by Uttam Kumar Patra
"""
})

# print(response)
# print(type(response))
# print(response.keys())

import json
print(json.dumps(response, indent=2))

{
  "key_themes": [
    "Powerful performance",
    "Gaming and productivity",
    "Battery life and fast charging",
    "Camera quality and zoom",
    "Low-light / night mode photography",
    "S-Pen functionality",
    "Device size and weight",
    "One UI bloatware",
    "High price"
  ],
  "summary": "The reviewer is very impressed with the Samsung Galaxy S25 Ultra\u2019s powerful performance, excellent battery life, fast charging, and especially the 200MP camera and night mode. They appreciate the S-Pen support but note that the phone is large and heavy for one-handed use, contains unnecessary Samsung bloatware, and is quite expensive.",
  "sentiment": "positive",
  "pros": [
    "Very powerful Snapdragon 8 Gen 3 processor",
    "Great for gaming, multitasking, and photo editing",
    "Long-lasting 5000mAh battery",
    "45W fast charging",
    "Excellent 200MP camera with impressive night mode",
    "Useful zoom up to about 30x",
    "S-Pen integration for notes and sketches"
  ]